# Mexico's Electric & Hybrid Vehicle Market
Run each cell in order to launch the dashboard.

In [1]:
!pip -q install streamlit plotly openpyxl


## Upload the Excel database
Choose `11_Veh_Hib_Elect_serie.xlsx`.

In [2]:
from google.colab import files
files.upload()

Saving 11_Veh_Hib_Elect_serie.xlsx to 11_Veh_Hib_Elect_serie.xlsx


{'11_Veh_Hib_Elect_serie.xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00b\xee\x9dh^\x01\x00\x00\x90\x04\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

## Create the dashboard app

In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
from pathlib import Path

st.set_page_config(page_title="Mexico Automotive Market", layout="wide")
st.title("Mexico's Electric & Hybrid Vehicle Market")
st.caption("Investment opportunities by vehicle technology and state")

@st.cache_data
def load_data(file):
    df = pd.read_excel(file, sheet_name="Publicado", header=4)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"Año":"YEAR", "Mes":"MONTH", "Entidad Federativa":"STATE",
        "Unidades Vehiculares Eléctricas":"EV", "Unidades Vehiculares Híbridas Plug-in1/":"PHEV",
        "Unidades Vehiculares Hibridas2/":"HEV"})
    df = df[["YEAR", "MONTH", "STATE", "EV", "PHEV", "HEV"]]
    df["YEAR"] = pd.to_numeric(df.YEAR, errors="coerce")
    df[["EV", "PHEV", "HEV"]] = df[["EV", "PHEV", "HEV"]].apply(pd.to_numeric, errors="coerce")
    df = df.dropna().query("STATE != 'No especificado'")
    months = dict(zip(["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"], range(1,13)))
    df["DATE"] = pd.to_datetime(dict(year=df.YEAR.astype(int), month=df.MONTH.map(months), day=1), errors="coerce")
    df = df.dropna(subset=["DATE"])
    df["TOTAL"] = df[["EV", "PHEV", "HEV"]].sum(axis=1)
    return df

file = next((p for p in ["11_Veh_Hib_Elect_serie.xlsx", "/content/11_Veh_Hib_Elect_serie.xlsx", "/mnt/data/11_Veh_Hib_Elect_serie.xlsx"] if Path(p).exists()), None)
if file is None:
    file = st.file_uploader("Upload the INEGI Excel file (.xlsx)", type="xlsx")
    if file is None: st.info("Upload 11_Veh_Hib_Elect_serie.xlsx to display the dashboard."); st.stop()
df = load_data(file)

st.sidebar.header("Filters")
year = st.sidebar.selectbox("Year", ["All"] + sorted(df.YEAR.astype(int).unique().tolist()))
state = st.sidebar.selectbox("State", ["All"] + sorted(df.STATE.unique()))
data = df if year == "All" and state == "All" else df.copy()
if year != "All": data = data[data.YEAR == int(year)]
if state != "All": data = data[data.STATE == state]

total, electric = data.TOTAL.sum(), data.EV.sum()
c1, c2, c3 = st.columns(3)
c1.metric("Total vehicles sold", f"{total:,.0f}")
c2.metric("Fully electric vehicles", f"{electric:,.0f}")
c3.metric("Fully electric share", f"{electric/total*100 if total else 0:.1f}%")
green, grey = "#009E73", "#B8B8B8"

def show(fig):
    fig.update_layout(template="plotly_white")
    fig.update_xaxes(showgrid=False); fig.update_yaxes(showgrid=False)
    st.plotly_chart(fig, use_container_width=True)

st.subheader("1. HEV accounts for over three-quarters of units sold")
tech = data[["EV", "PHEV", "HEV"]].sum().rename_axis("Technology").reset_index(name="Sales")
tech["Technology"] = tech.Technology.map({"EV":"Fully electric (EV)", "PHEV":"Plug-in hybrid (PHEV)", "HEV":"Hybrid (HEV)"})
show(px.bar(tech, x="Technology", y="Sales", color="Technology", text_auto=",.0f", color_discrete_map={"Fully electric (EV)":grey, "Plug-in hybrid (PHEV)":grey, "Hybrid (HEV)":green}).update_layout(showlegend=False, xaxis_title="", yaxis_title="Vehicles sold"))

st.subheader("2. HEV demand shows the strongest growth trajectory")
monthly = data.groupby("DATE")[ ["EV", "PHEV", "HEV"]].sum().reset_index().melt("DATE", var_name="Technology", value_name="Sales")
show(px.line(monthly, x="DATE", y="Sales", color="Technology", color_discrete_map={"EV":grey, "PHEV":grey, "HEV":green}).update_layout(xaxis_title="", yaxis_title="Vehicles sold", legend_title=""))

st.subheader("3. A small group of states leads overall vehicle demand")
regional = data.groupby("STATE").TOTAL.sum().nlargest(10).rename("Sales").reset_index()
top_states = regional.STATE.head(3).tolist()
regional["Group"] = regional.STATE.apply(lambda s: "Top 3 states" if s in top_states else "Other states")
show(px.bar(regional, x="Sales", y="STATE", orientation="h", color="Group", text_auto=",.0f", color_discrete_map={"Top 3 states":green, "Other states":"#C7CDD3"}).update_layout(showlegend=False, xaxis_title="Vehicles sold", yaxis_title="", yaxis={"categoryorder":"total ascending"}))

st.subheader("4. HEV leads demand across the highest-volume states")
state_mix = data[data.STATE.isin(top_states)].groupby("STATE")[ ["EV", "PHEV", "HEV"]].sum().reset_index().melt("STATE", var_name="Technology", value_name="Sales")
show(px.bar(state_mix, x="STATE", y="Sales", color="Technology", barmode="stack", color_discrete_map={"EV":grey, "PHEV":grey, "HEV":green}).update_layout(xaxis_title="", yaxis_title="Vehicles sold", legend_title=""))

st.subheader("Cleaned database")
st.dataframe(data, use_container_width=True, hide_index=True)
st.download_button("Download cleaned data", data.to_csv(index=False), "automotive_data.csv", "text/csv")
st.caption("Registrations show market demand, not profitability. Consider costs, incentives, infrastructure and competition before investing.")


Writing app.py


## Launch the dashboard
Keep this cell running and open the tunnel URL.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &
!./cloudflared tunnel --url http://localhost:8501

2026-11-27T06:58:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-27T06:58:20Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-27T06:58:24Z INF +--------------------------------------------------------------------------------------------+
2026-11-27T06:58:24Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-27T06:58:24Z INF |  https://firewall-liable-generations-through.trycloudf